# 12 - Análisis: Preguntas de Negocio respondidas con Gold

Trabajo Práctico - Pipeline ETL end-to-end (Bronze → Silver / DQL → Gold)

Este notebook reúne en un solo lugar **las preguntas de negocio** que responde el modelo Gold. Cada sección tiene la pregunta, la consulta sobre Gold, un gráfico, la tabla de datos y un espacio para la conclusión.

* Solo lee tablas **Gold**, más las tablas DQL en la sección de calidad. Así se evidencia que el análisis sale de datos ya validados.
* Requiere haber ejecutado el pipeline (`10_Orquestador_Pipeline` o el Job).
* Para la exposición: **Run all** y recorrer las secciones de arriba hacia abajo.

| Bloque | Preguntas | Tabla Gold |
|---|---|---|
| A. Contratación pública | P1 – P7 | `gold.de_fact_contratos` |
| B. Tejido empresarial | P8 – P12 | `gold.de_dim_empresas` |
| C. TRM y empresas | P13 | `gold.de_agg_indicadores_mes` |
| D. Cruce empresas–contratos | P14 | `gold.de_fact_contratos` + `gold.de_dim_empresas` |
| E. Calidad de datos | P15 | `dql.*` |

In [ ]:
%run ./00_Utilidades_Pipeline

In [ ]:
%python
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Paleta: una serie = un color; texto en tinta neutra; grilla recesiva
AZUL, NARANJA = "#2a78d6", "#eb6834"
TEXTO, TEXTO_2, GRILLA, FONDO = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRILLA, "axes.labelcolor": TEXTO_2, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlecolor": TEXTO, "xtick.color": TEXTO_2, "ytick.color": TEXTO_2, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})


def consulta(sql: str):
    """Ejecuta la consulta, muestra la tabla (vista accesible del gráfico) y devuelve un DataFrame de pandas."""
    df = spark.sql(sql)
    display(df)
    return df.toPandas()


def _ejes(ax, eje_valores: str, formato: str):
    fmt = mticker.FuncFormatter(lambda v, _: formato.format(v))
    if eje_valores == "x":
        ax.xaxis.set_major_formatter(fmt)
        ax.xaxis.grid(True, color=GRILLA, linewidth=0.8)
        ax.spines["left"].set_visible(False)
    else:
        ax.yaxis.set_major_formatter(fmt)
        ax.yaxis.grid(True, color=GRILLA, linewidth=0.8)
    ax.set_axisbelow(True)


def barras_h(pdf, categoria, valor, titulo, etiqueta, formato="{:,.0f}", largo=50):
    """Ranking: barras horizontales, la mayor arriba. Etiqueta de valor solo en la barra mayor."""
    if pdf.empty:
        print("Sin datos para graficar.")
        return
    datos = pdf.iloc[::-1]
    nombres = datos[categoria].astype(str).str.slice(0, largo)
    fig, ax = plt.subplots(figsize=(10, 0.42 * len(datos) + 1.3))
    ax.barh(nombres, datos[valor].astype(float), color=AZUL, height=0.62)
    ax.set_title(titulo, loc="left")
    ax.set_xlabel(etiqueta)
    _ejes(ax, "x", formato)
    mayor = float(pdf[valor].iloc[0])
    ax.annotate(formato.format(mayor), (mayor, len(datos) - 1), xytext=(4, 0), textcoords="offset points",
                va="center", color=TEXTO, fontsize=9)
    plt.tight_layout()
    plt.show()


def barras_v(pdf, categoria, valor, titulo, etiqueta, formato="{:,.0f}"):
    """Categorías ordenadas (rangos): barras verticales."""
    if pdf.empty:
        print("Sin datos para graficar.")
        return
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.bar(pdf[categoria].astype(str), pdf[valor].astype(float), color=AZUL, width=0.6)
    ax.set_title(titulo, loc="left")
    ax.set_ylabel(etiqueta)
    _ejes(ax, "y", formato)
    plt.tight_layout()
    plt.show()


def lineas(pdf, x, series: dict, titulo, etiqueta, formato="{:,.0f}"):
    """Serie de tiempo. series = {columna: nombre visible}; misma escala para todas (nunca doble eje)."""
    if pdf.empty:
        print("Sin datos para graficar.")
        return
    colores = [AZUL, NARANJA]
    fig, ax = plt.subplots(figsize=(10, 4.2))
    for i, (columna, nombre) in enumerate(series.items()):
        ax.plot(pdf[x], pdf[columna].astype(float), color=colores[i], linewidth=2, label=nombre)
    ax.set_title(titulo, loc="left")
    ax.set_ylabel(etiqueta)
    _ejes(ax, "y", formato)
    if len(series) > 1:
        ax.legend(frameon=False, loc="upper left")
    plt.tight_layout()
    plt.show()


MILES_MILLONES = 1e9  # los valores en pesos se muestran en miles de millones de COP

---
# A. Contratación pública (`gold.de_fact_contratos`)

In [ ]:
%python
display(spark.sql(f"""
    SELECT COUNT(*) AS contratos,
           ROUND(SUM(valor_contrato) / {MILES_MILLONES}, 1) AS valor_total_miles_millones_cop,
           COUNT(DISTINCT nit_entidad) AS entidades,
           COUNT(DISTINCT documento_proveedor_normalizado) AS proveedores,
           MIN(fecha_firma) AS primera_firma, MAX(fecha_firma) AS ultima_firma
    FROM {GOLD_FACT_CONTRATOS}
"""))

## P1. ¿Qué entidades concentran el mayor valor contratado?

In [ ]:
%python
p1 = consulta(f"""
    SELECT nombre_entidad,
           COUNT(*) AS contratos,
           CAST(ROUND(SUM(valor_contrato) / {MILES_MILLONES}, 2) AS DOUBLE) AS valor_miles_millones_cop,
           CAST(ROUND(100 * SUM(valor_contrato) / SUM(SUM(valor_contrato)) OVER (), 1) AS DOUBLE) AS pct_del_total
    FROM {GOLD_FACT_CONTRATOS}
    GROUP BY nombre_entidad
    ORDER BY valor_miles_millones_cop DESC
    LIMIT 10
""")
barras_h(p1, "nombre_entidad", "valor_miles_millones_cop",
         "Top 10 entidades por valor contratado", "Miles de millones de COP", "{:,.1f}")

**Conclusión P1:** _(completar después de ejecutar: ¿cuánto del total concentran las 10 primeras entidades?)_

## P2. ¿Qué departamentos reciben más contratación?

In [ ]:
%python
p2 = consulta(f"""
    SELECT departamento,
           COUNT(*) AS contratos,
           CAST(ROUND(SUM(valor_contrato) / {MILES_MILLONES}, 2) AS DOUBLE) AS valor_miles_millones_cop
    FROM {GOLD_FACT_CONTRATOS}
    GROUP BY departamento
    ORDER BY valor_miles_millones_cop DESC
    LIMIT 10
""")
barras_h(p2, "departamento", "valor_miles_millones_cop",
         "Top 10 departamentos por valor contratado", "Miles de millones de COP", "{:,.1f}")

**Conclusión P2:** _(completar)_

## P3. ¿Qué tanto se contrata de forma directa (sin competencia)?

La contratación directa se adjudica sin un proceso competitivo. Una proporción alta es un indicador clásico de riesgo en compras públicas.

In [ ]:
%python
p3 = consulta(f"""
    SELECT modalidad_contratacion,
           COUNT(*) AS contratos,
           CAST(ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS DOUBLE) AS pct_contratos,
           CAST(ROUND(100 * SUM(valor_contrato) / SUM(SUM(valor_contrato)) OVER (), 1) AS DOUBLE) AS pct_valor
    FROM {GOLD_FACT_CONTRATOS}
    GROUP BY modalidad_contratacion
    ORDER BY contratos DESC
""")
barras_h(p3, "modalidad_contratacion", "pct_contratos",
         "Contratos por modalidad de contratación", "% de los contratos", "{:,.1f}%")

display(spark.sql(f"""
    SELECT CAST(ROUND(100 * AVG(CASE WHEN es_contratacion_directa THEN 1 ELSE 0 END), 1) AS DOUBLE) AS pct_contratos_directos,
           CAST(ROUND(100 * SUM(CASE WHEN es_contratacion_directa THEN valor_contrato ELSE 0 END) / SUM(valor_contrato), 1) AS DOUBLE) AS pct_valor_directo
    FROM {GOLD_FACT_CONTRATOS}
"""))

**Conclusión P3:** _(completar: % de contratos y % del valor adjudicado por contratación directa)_

## P4. ¿Qué participación tienen las pymes: en número de contratos y en valor?

Si las pymes tienen una proporción alta de contratos pero baja del valor, reciben muchos contratos pequeños.

In [ ]:
%python
p4 = consulta(f"""
    SELECT 'Contratos' AS medida,
           CAST(ROUND(100 * AVG(CASE WHEN es_pyme THEN 1 ELSE 0 END), 1) AS DOUBLE) AS pct_pyme
    FROM {GOLD_FACT_CONTRATOS}
    UNION ALL
    SELECT 'Valor contratado',
           CAST(ROUND(100 * SUM(CASE WHEN es_pyme THEN valor_contrato ELSE 0 END) / SUM(valor_contrato), 1) AS DOUBLE)
    FROM {GOLD_FACT_CONTRATOS}
""")
barras_h(p4, "medida", "pct_pyme", "Participación de pymes en la contratación", "% que corresponde a pymes", "{:,.1f}%")

**Conclusión P4:** _(completar)_

## P5. ¿Qué tan ejecutados están los contratos y cuántos se prorrogan?

`porcentaje_pagado` = valor pagado / valor del contrato. Silver garantiza que está entre 0 % y 100 %.

In [ ]:
%python
p5 = consulta(f"""
    SELECT CASE WHEN porcentaje_pagado = 0 THEN '1. 0 %'
                WHEN porcentaje_pagado <= 25 THEN '2. 1-25 %'
                WHEN porcentaje_pagado <= 50 THEN '3. 26-50 %'
                WHEN porcentaje_pagado <= 75 THEN '4. 51-75 %'
                WHEN porcentaje_pagado < 100 THEN '5. 76-99 %'
                ELSE '6. 100 %' END AS rango_pagado,
           COUNT(*) AS contratos
    FROM {GOLD_FACT_CONTRATOS}
    GROUP BY 1
    ORDER BY 1
""")
barras_v(p5, "rango_pagado", "contratos", "Contratos según el porcentaje pagado", "Contratos")

display(spark.sql(f"""
    SELECT CAST(ROUND(AVG(porcentaje_pagado), 1) AS DOUBLE) AS pct_pagado_promedio,
           SUM(CASE WHEN tiene_adicion_tiempo THEN 1 ELSE 0 END) AS contratos_con_adicion_tiempo,
           CAST(ROUND(100 * AVG(CASE WHEN tiene_adicion_tiempo THEN 1 ELSE 0 END), 1) AS DOUBLE) AS pct_con_adicion_tiempo,
           CAST(ROUND(AVG(duracion_dias), 0) AS DOUBLE) AS duracion_promedio_dias
    FROM {GOLD_FACT_CONTRATOS}
"""))

**Conclusión P5:** _(completar)_

## P6. ¿Qué proveedores concentran más valor contratado?

In [ ]:
%python
p6 = consulta(f"""
    SELECT proveedor_adjudicado,
           COUNT(*) AS contratos,
           CAST(ROUND(SUM(valor_contrato) / {MILES_MILLONES}, 2) AS DOUBLE) AS valor_miles_millones_cop
    FROM {GOLD_FACT_CONTRATOS}
    GROUP BY proveedor_adjudicado
    ORDER BY valor_miles_millones_cop DESC
    LIMIT 10
""")
barras_h(p6, "proveedor_adjudicado", "valor_miles_millones_cop",
         "Top 10 proveedores por valor adjudicado", "Miles de millones de COP", "{:,.1f}")

# Concentración: qué parte del valor se lleva el 1 % de proveedores con más valor
display(spark.sql(f"""
    WITH por_proveedor AS (
        SELECT documento_proveedor_normalizado, SUM(valor_contrato) AS valor
        FROM {GOLD_FACT_CONTRATOS} GROUP BY documento_proveedor_normalizado),
    ranking AS (
        SELECT valor, PERCENT_RANK() OVER (ORDER BY valor DESC) AS pr FROM por_proveedor)
    SELECT COUNT(*) AS proveedores,
           CAST(ROUND(100 * SUM(CASE WHEN pr < 0.01 THEN valor ELSE 0 END) / SUM(valor), 1) AS DOUBLE) AS pct_valor_top_1pct_proveedores
    FROM ranking
"""))

**Conclusión P6:** _(completar: ¿qué % del valor se lleva el 1 % de proveedores más grandes?)_

## P7. ¿Cómo evoluciona la contratación mes a mes (en COP y en USD)?

Son dos medidas en escalas distintas, así que van en dos gráficos y no en uno con doble eje. El valor en dólares solo existe para los meses cubiertos por la TRM.

In [ ]:
%python
p7 = consulta(f"""
    SELECT anio_mes, contratos_firmados,
           CAST(valor_contratado_cop / {MILES_MILLONES} AS DOUBLE) AS valor_miles_millones_cop,
           CAST(valor_contratado_usd / 1e6 AS DOUBLE) AS valor_millones_usd
    FROM {GOLD_AGG_MES}
    WHERE anio_mes >= '2020-01-01' AND contratos_firmados > 0
    ORDER BY anio_mes
""")
lineas(p7, "anio_mes", {"contratos_firmados": "Contratos"}, "Contratos firmados por mes", "Contratos")
lineas(p7.dropna(subset=["valor_millones_usd"]), "anio_mes", {"valor_millones_usd": "USD"},
       "Valor contratado por mes (con TRM del día de firma)", "Millones de USD", "{:,.1f}")

**Conclusión P7:** _(completar)_

---
# B. Tejido empresarial (`gold.de_dim_empresas`)

In [ ]:
%python
display(spark.sql(f"""
    SELECT COUNT(*) AS empresas,
           SUM(CASE WHEN es_activa THEN 1 ELSE 0 END) AS activas,
           SUM(CASE WHEN es_cancelada THEN 1 ELSE 0 END) AS canceladas,
           COUNT(DISTINCT camara_comercio) AS camaras_de_comercio,
           COUNT(DISTINCT codigo_ciiu_principal) AS actividades_ciiu
    FROM {GOLD_DIM_EMPRESAS}
"""))

## P8. ¿Qué actividades económicas concentran más empresas activas?

Se agrupa por **división CIIU** (2 dígitos) para que el ranking sea legible.

In [ ]:
%python
p8 = consulta(f"""
    SELECT CONCAT(division_ciiu_principal, ' · ', MIN(descripcion_ciiu_principal)) AS actividad,
           SUM(CASE WHEN es_activa THEN 1 ELSE 0 END) AS empresas_activas
    FROM {GOLD_DIM_EMPRESAS}
    GROUP BY division_ciiu_principal
    ORDER BY empresas_activas DESC
    LIMIT 10
""")
barras_h(p8, "actividad", "empresas_activas", "Divisiones CIIU con más empresas activas", "Empresas activas")

**Conclusión P8:** _(completar)_

## P9. ¿Qué actividades tienen la mayor tasa de cancelación?

Solo se incluyen divisiones con al menos 100 empresas, para que la tasa sea estable.

In [ ]:
%python
p9 = consulta(f"""
    SELECT CONCAT(division_ciiu_principal, ' · ', MIN(descripcion_ciiu_principal)) AS actividad,
           COUNT(*) AS empresas,
           SUM(CASE WHEN es_cancelada THEN 1 ELSE 0 END) AS canceladas,
           CAST(ROUND(100 * AVG(CASE WHEN es_cancelada THEN 1 ELSE 0 END), 1) AS DOUBLE) AS pct_canceladas
    FROM {GOLD_DIM_EMPRESAS}
    GROUP BY division_ciiu_principal
    HAVING COUNT(*) >= 100
    ORDER BY pct_canceladas DESC
    LIMIT 10
""")
barras_h(p9, "actividad", "pct_canceladas", "Divisiones CIIU con mayor tasa de cancelación", "% de empresas canceladas", "{:,.1f}%")

**Conclusión P9:** _(completar)_

## P10. ¿Bajo qué forma jurídica se organizan las empresas?

In [ ]:
%python
p10 = consulta(f"""
    SELECT organizacion_juridica, COUNT(*) AS empresas,
           CAST(ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS DOUBLE) AS pct
    FROM {GOLD_DIM_EMPRESAS}
    GROUP BY organizacion_juridica
    ORDER BY empresas DESC
    LIMIT 8
""")
barras_h(p10, "organizacion_juridica", "empresas", "Empresas por organización jurídica", "Empresas", largo=45)

**Conclusión P10:** _(completar)_

## P11. ¿Cuántas empresas se crean y se cancelan cada mes?

In [ ]:
%python
p11 = consulta(f"""
    SELECT anio_mes, empresas_matriculadas, empresas_canceladas, saldo_neto_empresas
    FROM {GOLD_AGG_MES}
    WHERE anio_mes >= '2015-01-01'
    ORDER BY anio_mes
""")
lineas(p11, "anio_mes", {"empresas_matriculadas": "Matriculadas", "empresas_canceladas": "Canceladas"},
       "Empresas matriculadas y canceladas por mes", "Empresas")

**Conclusión P11:** _(completar)_

## P12. ¿Cuánto duran las empresas que se cancelan?

Años entre la matrícula y la cancelación, por forma jurídica (con al menos 30 cancelaciones).

In [ ]:
%python
p12 = consulta(f"""
    SELECT organizacion_juridica,
           COUNT(*) AS canceladas,
           CAST(ROUND(AVG(anios_hasta_cancelacion), 1) AS DOUBLE) AS anios_promedio,
           CAST(PERCENTILE(anios_hasta_cancelacion, 0.5) AS DOUBLE) AS anios_mediana
    FROM {GOLD_DIM_EMPRESAS}
    WHERE anios_hasta_cancelacion IS NOT NULL
    GROUP BY organizacion_juridica
    HAVING COUNT(*) >= 30
    ORDER BY anios_mediana DESC
""")
barras_h(p12, "organizacion_juridica", "anios_mediana",
         "Vida mediana de las empresas canceladas", "Años desde la matrícula hasta la cancelación", "{:,.1f}", largo=45)

**Conclusión P12:** _(completar)_

---
# C. TRM y empresas (`gold.de_agg_indicadores_mes`)

## P13. ¿Influye la TRM en el cierre de empresas?

La TRM (COP por USD) y las cancelaciones tienen escalas distintas, por eso van en **dos paneles con el mismo eje de tiempo**, no en un gráfico con doble eje. La correlación es **exploratoria**: Bronze solo tiene unos 12 meses de TRM, y correlación no implica causalidad.

In [ ]:
%python
p13 = consulta(f"""
    SELECT anio_mes, CAST(trm_promedio AS DOUBLE) AS trm_promedio, variacion_trm_pct,
           empresas_canceladas, empresas_matriculadas
    FROM {GOLD_AGG_MES}
    WHERE trm_promedio IS NOT NULL
    ORDER BY anio_mes
""")

if not p13.empty:
    fig, (arriba, abajo) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    arriba.plot(p13["anio_mes"], p13["trm_promedio"], color=AZUL, linewidth=2)
    arriba.set_title("TRM promedio mensual (COP por USD)", loc="left")
    _ejes(arriba, "y", "{:,.0f}")
    abajo.bar(p13["anio_mes"], p13["empresas_canceladas"].astype(float), color=AZUL, width=20)
    abajo.set_title("Empresas canceladas por mes", loc="left")
    _ejes(abajo, "y", "{:,.0f}")
    plt.tight_layout()
    plt.show()

display(spark.sql(f"""
    SELECT COUNT(*) AS meses_con_trm,
           ROUND(TRY_DIVIDE(COVAR_POP(trm_promedio, empresas_canceladas),
                            STDDEV_POP(trm_promedio) * STDDEV_POP(empresas_canceladas)), 3) AS corr_trm_cancelaciones,
           ROUND(TRY_DIVIDE(COVAR_POP(trm_promedio, empresas_matriculadas),
                            STDDEV_POP(trm_promedio) * STDDEV_POP(empresas_matriculadas)), 3) AS corr_trm_matriculas
    FROM {GOLD_AGG_MES}
    WHERE trm_promedio IS NOT NULL
"""))

**Conclusión P13:** _(completar. Guía de lectura: |r| < 0,3 es una relación débil, 0,3–0,7 moderada y > 0,7 fuerte. Con unos 12 meses, cualquier valor es exploratorio.)_

---
# D. Cruce empresas – contratos

## P14. ¿Cuántos proveedores del Estado están registrados en el RUES y a qué se dedican?

El enlace se hace por documento (NIT o identificación) entre `gold.de_fact_contratos` y `gold.de_dim_empresas`.

In [ ]:
%python
display(spark.sql(f"""
    SELECT tipo_documento_proveedor,
           COUNT(*) AS contratos,
           SUM(CASE WHEN proveedor_registrado_rues THEN 1 ELSE 0 END) AS contratos_con_proveedor_en_rues,
           CAST(ROUND(100 * AVG(CASE WHEN proveedor_registrado_rues THEN 1 ELSE 0 END), 2) AS DOUBLE) AS pct
    FROM {GOLD_FACT_CONTRATOS}
    GROUP BY tipo_documento_proveedor
    ORDER BY contratos DESC
"""))

p14 = consulta(f"""
    SELECT CONCAT(e.division_ciiu_principal, ' · ', MIN(e.descripcion_ciiu_principal)) AS actividad_proveedor,
           COUNT(*) AS contratos,
           CAST(ROUND(SUM(f.valor_contrato) / {MILES_MILLONES}, 2) AS DOUBLE) AS valor_miles_millones_cop
    FROM {GOLD_FACT_CONTRATOS} f
    JOIN {GOLD_DIM_EMPRESAS} e ON e.llave_empresa = f.llave_empresa_proveedor
    GROUP BY e.division_ciiu_principal
    ORDER BY valor_miles_millones_cop DESC
    LIMIT 10
""")
barras_h(p14, "actividad_proveedor", "valor_miles_millones_cop",
         "Actividad económica de los proveedores registrados en RUES", "Miles de millones de COP", "{:,.2f}")

**Conclusión P14:** _(completar. Contexto: la mayoría de los proveedores son personas naturales con cédula, y el RUES cargado es una muestra, así que la cobertura del cruce es baja.)_

---
# E. Calidad de datos

## P15. ¿Cuántos registros se rechazaron y por qué?

Un registro puede incumplir varias reglas, así que la suma por regla puede superar el total de rechazados.

In [ ]:
%python
fuentes_sql = " UNION ALL ".join(
    f"SELECT '{nombre}' AS fuente, (SELECT COUNT(*) FROM {f['bronze']}) AS bronze, "
    f"(SELECT COUNT(*) FROM {f['silver']}) AS silver, (SELECT COUNT(*) FROM {f['dql']}) AS dql"
    for nombre, f in FUENTES.items()
)
p15a = consulta(f"""
    SELECT fuente, bronze, silver, dql,
           CAST(ROUND(100 * dql / bronze, 2) AS DOUBLE) AS pct_rechazado
    FROM ({fuentes_sql})
    ORDER BY pct_rechazado DESC
""")
barras_h(p15a, "fuente", "pct_rechazado", "Porcentaje de registros de Bronze rechazados", "% rechazado", "{:,.2f}%")

reglas_sql = " UNION ALL ".join(
    f"SELECT '{nombre}' AS fuente, regla FROM {f['dql']} LATERAL VIEW EXPLODE(SPLIT(_motivo_rechazo, '; ')) r AS regla"
    for nombre, f in FUENTES.items()
)
p15b = consulta(f"""
    SELECT CONCAT(fuente, ' · ', regla) AS fuente_regla, COUNT(*) AS registros
    FROM ({reglas_sql})
    GROUP BY fuente, regla
    ORDER BY registros DESC
""")
barras_h(p15b, "fuente_regla", "registros", "Registros rechazados por regla de calidad", "Registros")

**Conclusión P15:** _(completar)_